# 🌍 Pipeline de Georreferenciación y Calidad de Datos IPS
**Proyecto:** Optimización de Red de Prestadores de Salud  
**Objetivo:** Normalizar, enriquecer y geolocalizar direcciones de sedes médicas utilizando algoritmos híbridos (Python + Base Oficial REPS + Inteligencia Artificial).

### 🛠️ 1. Configuración e Importación de Librerías
Se cargan las herramientas necesarias para manipulación de datos (`pandas`), procesamiento de texto (`regex`, `unidecode`), conexión a APIs de geolocalización (`geopy`, `ArcGIS`) y el cliente de Inteligencia Artificial (`OpenAI`).

In [ ]:
# Libraries
import sys
import pandas as pd
from typing import List, Dict, Tuple
import os
import glob
from pathlib import Path
import numpy as np
import json
from unidecode import unidecode
import geopy
from geopy.geocoders import ArcGIS
from geopy.distance import geodesic
from typing import Optional, Union
import time
import re
import logging
from enum import Enum
from collections import Counter
import unicodedata
from enum import Enum
from typing import List, Union
from IPython.display import display  # Necesario para mostrar bien los DataFrames
from openai import OpenAI
import openai
import geocoder
from geopy.distance import geodesic
import matplotlib.pyplot as plt

### 📥 2. Ingesta y Pre-procesamiento de Datos
Cargamos la base maestra de prestadores y aplicamos un primer filtro de **Limpieza Negativa**.
* **Objetivo:** Eliminar ruido evidente que no corresponde a sedes físicas.
* **Filtros aplicados:** Se descartan registros que contengan términos como *"VIRTUAL"*, *"DOMICILIARIA"*, *"POR CONFIRMAR"* o *"ORDEN DE COMPRA"*.

In [ ]:
ruta = "/Users/dvarela/Downloads/"

In [ ]:
prestadores = pd.read_excel(ruta + "Prestadores (MAPA).xlsx", index_col=0)

In [19]:
# Crear la copia de seguridad (Deep Copy)
df = prestadores.copy()

In [ ]:
# 1. Aseguramos que la columna sea texto para evitar errores con nulos
df["Prestador"] = df["Prestador"].astype(str)
df["Tipo Prestador"] = df["Tipo Prestador"].astype(str)
# 2. Definimos las condiciones a eliminar
# Usamos case=False para que no importen las mayúsculas/minúsculas
# na=False ignora los valores vacíos para que no den error

condicion_eliminar = (
    # Elimina exactos o aproximados de "ORDEN DE COMPRA..."
    df["Prestador"].str.contains("ORDEN DE COMPRA PUNTUAL", case=False, na=False)
    |
    # Elimina el caso específico que pediste (y maneja si falta el espacio entre INICIAL y POR)
    df["Prestador"].str.contains(
        "IPS DE ATENCION INICIAL.*POR CONFIRMAR", case=False, regex=True, na=False
    )
    |
    # Elimina cualquier registro que contenga "ATENCION DOMICILIARIA" en cualquier parte
    df["Prestador"].str.contains("ATENCION DOMICILIARIA", case=False, na=False)
    |
    # Elimina cualquier registro que contenga "VIRTUAL" en cualquier parte
    df["Prestador"].str.contains("VIRTUAL", case=False, na=False)
    |
    # SUGERENCIA ADICIONAL: Viendo tu imagen, hay muchos "POR CONFIRMAR" genéricos.
    # Esta línea extra te limpiaría "IPS DE ATENCION SIN CONFIRMAR", "PRESTADOR POR CONFIRMAR", etc.
    df["Prestador"].str.contains("POR CONFIRMAR", case=False, na=False)
    |
    # Esta línea limpiaría Tipo de prestador "Establecimientos", etc.
    df["Tipo Prestador"].str.contains("Establecimientos", case=False, na=False)
)

# 3. Aplicamos el filtro INVERSO (~)
# "Quédate con las filas que NO (~) cumplan la condición de eliminar"
df = df[~condicion_eliminar]

# 4. Verificación
print(f"Registros después de la limpieza: {len(df)}")

Registros después de la limpieza: 36149


In [ ]:
# 1. Definimos las columnas que determinan si un registro es único.
# Si dos filas tienen los mismos valores en ESTAS columnas, se consideran duplicadas.
cols_para_duplicados = [
    "Sucursal Prestador",
    "Documento Prestador",
    "Departamento",
    "Municipio",
]

# 2. Quitamos espacios en blanco al inicio/final antes de comparar
for col in cols_para_duplicados:
    # Solo aplicamos strip si la columna es de tipo texto (object)
    if df[col].dtype == "object":
        df[col] = df[col].str.strip()

# 3. Ejecutamos la eliminación de duplicados
# keep='first': Mantiene la primera aparición y borra las repetidas.
# inplace=True: Modifica el DataFrame directamente (o usa una nueva variable si prefieres)
df.drop_duplicates(subset=cols_para_duplicados, keep="first", inplace=True)

# 4. Reseteamos el índice para que la numeración de filas quede ordenada
df.reset_index(drop=True, inplace=True)

# Verificación
print(f"Registros únicos restantes: {len(df)}")

Registros únicos restantes: 2031


### 🧠 3. Motor de Estandarización de Direcciones (Algoritmo Maestro v12)
Este es el núcleo lógico del procesamiento. Se define una función personalizada que limpia y estandariza las direcciones crudas.

**Lógica del Algoritmo:**
1.  **Stop Words:** Elimina palabras "basura" que confunden al geocodificador (ej: *"Frente a", "Consultorio", "Barrio"*).
2.  **Mapeo de Nomenclatura:** Convierte variaciones (ej: *"CRA", "K", "CARRERA"*) al estándar oficial (*"KR"*).
3.  **Recursividad:** Si la dirección empieza con el nombre del edificio, el algoritmo busca recursivamente dentro del texto hasta encontrar el patrón de vía (`Calle/Carrera` + `Número`).
4.  **Salvavidas Rural:** Detecta y preserva direcciones no urbanas válidas (ej: *"Vereda", "Km", "Vía"*).

In [ ]:
# =============================================================================
# 1. LISTA NEGRA (STOP WORDS) - CURADA Y SEGURA
# =============================================================================
STOP_WORDS_LIST = [
    # 1. Inmuebles y Ubicaciones
    "APARTAMENTO",
    "APTO",
    "APT",
    "CASA",
    "TORRE",
    "T",
    "TR",
    "OFICINA",
    "OF",
    "CONSULTORIO",
    "CONS",
    "CONSU",
    "CS",
    "CNO",  # QUITAMOS "CON" (peligroso para intersecciones)
    "BLOQUE",
    "BL",
    "UNIDAD",
    "INT",
    "INTERIOR",
    "PISO",
    "PIS",
    "PI",
    "PA",
    "P",
    "LOCAL",
    "LC",
    "L",
    "ACCESO",
    "ACC",
    "ENTRADA",
    "HANGAR",
    "BODEGA",
    "AEROPUERTO",
    # 2. Zonas y Conjuntos
    "BARRIO",
    "BR",
    "BRR",
    "BRRIO",
    "URBANIZACION",
    "URB",
    "CONJUNTO",
    "CONJ",  # QUITAMOS "B" (es parte de nomenclatura)
    "ETAPA",
    "SECTOR",
    "SEC",
    "ZONA",
    "CENTRO COMERCIAL",
    "CC",
    "MALL",
    "PLAZA",
    "PARQUE",
    "COND",
    "CONDOMINIO",
    "VENEZIA",
    "BRISAS",
    "RIO",
    # 3. Instituciones y Edificios
    "AVENIDA",
    "AV",
    "EDIFICIO",
    "ED",
    "EDIF",
    "CLINICA",
    "CLIN",
    "HOSPITAL",
    "IPS",
    "ESE",
    "LINICA",
    "SOMA",
    "JASBAN",
    "PORTO",
    "PORTOAZ",
    "MEDICA",
    "VIVENZA",
    "NOVA",
    "NOVACENTRO",
    "INNOVO",
    "COLEGIO",
    "U",
    "UNV",
    "UNIVERSIDAD",
    "CORREDOR",
    "UMBRI",
    "COMUNCA",
    "PRESENTACION",
    # 4. Ciudades y Lugares (Para limpiar el final)
    "ENVIGADO",
    "CHIA",
    "CAJICA",
    "BUCARAMANGA",
    "MEDELLIN",
    "CALI",
    "BOGOTA",
    "CARTAGENA",
    "BARRANQUILLA",
    "SOACHA",
    "BELLO",
    "ITAGUI",
    "PEREIRA",
    "MANIZALES",
    "SEGOVIA",
    "MABEL OTERO",
    "CONEX CONS",
    "AEROPUERTO E",
    # 5. Palabras de relleno
    "EL",
    "LA",
    "LOS",
    "LAS",
    "DEL",
    "DE",
    "FRENTE",
    "DIAGONAL A",
    "AL LADO",
    "ESQUINA",
    "ESQ",
    "COMUNA",
    "COM",
    "LOCALIDAD",
    "LOC",
    "CASCO",
    "URBANO",
    "PUERTA",
]

# Compilamos el Regex
STOP_WORDS_RE = re.compile(
    r"\b(" + "|".join(STOP_WORDS_LIST) + r")\b", flags=re.IGNORECASE
)


def estandarizador_maestro_v12(direccion, es_recursivo=False):
    # 1. Validación inicial
    if pd.isna(direccion) or str(direccion).strip() == "":
        return "Error: Vacío"

    # Limpieza inicial de comillas y formato IA
    text_raw = str(direccion).strip().replace('"', "").replace("'", "").rstrip(".")
    # Estrategia de la Coma (Si la IA devuelve "Dir, Ciudad")
    if "," in text_raw:
        text_raw = text_raw.split(",")[0]

    text_raw = text_raw.upper().strip()

    # 2. REGLA DE MUERTE SÚBITA
    if (
        text_raw.startswith("ATIENDE")
        or text_raw.startswith("SEDE ADMIN")
        or "NO EXISTE SEDE" in text_raw
    ):
        return "Error: No es dirección física"

    # 3. Pre-limpieza técnica
    text_raw = (
        text_raw.replace("N°", " ")
        .replace("Nº", " ")
        .replace("°", " ")
        .replace("º", " ")
    )
    text = unidecode(text_raw).strip()

    # Limpieza de "NO" (Número)
    text = re.sub(r"\bNO\b", " ", text)

    text = (
        text.replace(" N ", " ").replace("#", " ").replace("-", " ").replace(".", " ")
    )

    # Separar letras y números
    text = re.sub(r"([A-Z]+)(\d+)", r"\1 \2", text)
    text = re.sub(r"(\d+)([A-Z]+)", r"\1 \2", text)

    # Diccionario Maestro (Actualizado con Hallazgos)
    mapeo = {
        "CALLE": "CL",
        "CLL": "CL",
        "CL": "CL",
        "CALEL": "CL",
        "C": "CL",
        "CALL": "CL",
        "CARRERA": "KR",
        "CRA": "KR",
        "KRA": "KR",
        "CR": "KR",
        "KR": "KR",
        "K": "KR",
        "CARR": "KR",
        "CRRA": "KR",
        "CRR": "KR",  # Agregado CRR
        "DIAGONAL": "DG",
        "DIAG": "DG",
        "DG": "DG",
        "DIA": "DG",
        "TRANSVERSAL": "TV",
        "TRANS": "TV",
        "TR": "TV",
        "TV": "TV",
        "TRV": "TV",
        "TRANV": "TV",
        "TRANVS": "TV",
        "TRANSV": "TV",
        "TRAS": "TV",
        "AVENIDA": "AV",
        "AV": "AV",
        "AVE": "AV",
        "AVDA": "AV",
        "CORREDOR": "AV",
        "CORRED": "AV",  # Agregado CORREDOR -> AV
        "AC": "AV CL",
        "AK": "AV KR",
        "AUTOPISTA": "AU",
        "AUTOP": "AU",
        "AUT": "AU",
        "CIRCULAR": "CQ",
        "CIR": "CQ",
        "CQ": "CQ",
        "VIA": "VIA",
        "VARIANTE": "VIA",
        # Rurales
        "KM": "KM",
        "KILOMETRO": "KM",
        "VEREDA": "VDA",
        "VDA": "VDA",
        "CORREGIMIENTO": "CORREGIMIENTO",
        "FINCA": "FINCA",
        "HACIENDA": "HDA",
        "MANZANA": "MZ",
        "MZ": "MZ",
        "LOTE": "LT",
    }

    text = re.sub(r"\s+", " ", text).strip()

    # --- ESTRATEGIA 0: SALVAVIDAS RURAL ---
    palabras_salvavidas = [
        "KM ",
        "VIA ",
        "VEREDA ",
        "VDA ",
        "CORREGIMIENTO ",
        "LOTE ",
        "FINCA ",
    ]
    if any(k in text for k in palabras_salvavidas):
        return text

    parts = text.split()
    if not parts:
        return "Error: Sin info"
    primer_token = parts[0]

    # --- ESTRATEGIA 1: URBANA ESTÁNDAR ---
    if primer_token in mapeo:
        tipo_via = mapeo[primer_token]
        resto = " ".join(parts[1:])

        # EL GRAN CORTE (STOP WORDS)
        match = STOP_WORDS_RE.search(resto)
        if match:
            # Cortamos todo desde la palabra prohibida
            resto = resto[: match.start()].strip()

        # Limpieza final
        resto = re.sub(r"\s+", " ", resto).strip()

        # Validación
        if not any(char.isdigit() for char in resto) and len(resto) > 0:
            # Si es una vía sola (ej: KR 51) es válido.
            # Si es texto basura largo, es error.
            if len(resto) > 3 and not "AV" in tipo_via:
                return "Error: Vía sin número"

        return f"{tipo_via} {resto}".strip()

    # --- ESTRATEGIA 2: RECURSIVIDAD (Búsqueda interna) ---
    elif not es_recursivo:
        for i, token in enumerate(parts):
            if i > 0 and token in mapeo:
                if i + 1 < len(parts):
                    siguiente = parts[i + 1]
                    if siguiente.isdigit() or (
                        len(siguiente) > 0 and siguiente[0].isdigit()
                    ):
                        nuevo_inicio = " ".join(parts[i:])
                        return estandarizador_maestro_v12(
                            nuevo_inicio, es_recursivo=True
                        )

    # --- ESTRATEGIA 3: NOMBRES DE AVENIDAS ---
    if primer_token in ["AV", "AVENIDA"]:
        return text

    if primer_token.isdigit():
        return "Error: Empieza con número"

    # --- ESTRATEGIA 4: INTENTO DE LIMPIEZA FINAL SI NO ENCONTRÓ VÍA ---
    # Ej: "CLINICA LA PRESENTACION CONS 327" -> "CLINICA LA PRESENTACION"
    # Quitamos la basura del final aunque no hayamos encontrado "CL" o "KR"
    match = STOP_WORDS_RE.search(text)
    if match:
        text_limpio = text[: match.start()].strip()
        if len(text_limpio) > 4:
            return text_limpio

    return "Error: Formato Desconocido"


# --- PRUEBA CON TUS CASOS CRÍTICOS ---
print("--- TEST FINAL DE SEGURIDAD ---")
casos = [
    "KR 7 B 12 106",  # El caso de la "B" (Debe salir completo)
    "CRA 7 CON CALLE 14 ESQUINA",  # El caso de "CON" (No debe cortar en CON)
    "CORRED UNIVE 1 850 CONSU 708",  # El caso de CORREDOR y CONSU
    "CRR 9 A 11 57",  # El caso de CRR
    "CLIN LA PRESENTACION CONS 327",  # El caso sin vía inicial
]

for c in casos:
    print(f"{c:30} -> {estandarizador_maestro_v12(c)}")

--- TEST FINAL DE SEGURIDAD ---
KR 7 B 12 106                  -> KR 7 B 12 106
CRA 7 CON CALLE 14 ESQUINA     -> KR 7 CON CALLE 14
CORRED UNIVE 1 850 CONSU 708   -> AV UNIVE 1 850
CRR 9 A 11 57                  -> KR 9 A 11 57
CLIN LA PRESENTACION CONS 327  -> Error: Formato Desconocido


In [ ]:
# APLICAMOS LA FUNCIÓN
df["Direccion_Normalizada"] = df["Direccion Domicilio"].apply(
    estandarizador_maestro_v12
)

In [ ]:
df["Direccion_Normalizada"]

0             CL 1 4 45
1          CL 27 7 B 51
2       KR 13 W 16 A 49
3         KR 18 F 43 15
4             CL 5 7 08
             ...       
2026        CL 27 35 16
2027          KR 47 439
2028      KR 18 D 22 45
2029        KR 14 22 45
2030        CL 30 19 20
Name: Direccion_Normalizada, Length: 2031, dtype: object

In [ ]:
# Filtramos los que fallaron
df_errores = df[df["Direccion_Normalizada"].str.contains("Error", na=False)]

print("--- Conteo de tipos de Error ---")
print(df_errores["Direccion_Normalizada"].value_counts())

print("\n--- Ejemplos de direcciones que fallaron ---")
print(df_errores[["Direccion Domicilio", "Direccion_Normalizada"]].head(10))

--- Conteo de tipos de Error ---
Direccion_Normalizada
Error: Vacío                     75
Error: Formato Desconocido       35
Error: Vía sin número            12
Error: No es dirección física     1
Name: count, dtype: int64

--- Ejemplos de direcciones que fallaron ---
                Direccion Domicilio          Direccion_Normalizada
13   CLIN LA PRESENTACION  CONS 327     Error: Formato Desconocido
50   C.C. PASEO LA CASTELLANA L 1-1          Error: Vía sin número
124          ATIENDE EN BUCARAMANGA  Error: No es dirección física
175                             NaN                   Error: Vacío
176                             NaN                   Error: Vacío
221                             NaN                   Error: Vacío
234                             NaN                   Error: Vacío
237                             NaN                   Error: Vacío
238                             NaN                   Error: Vacío
239                             NaN                   Error

In [ ]:
def rescatar_rurales(fila):
    dir_norm = fila["Direccion_Normalizada"]
    dir_orig = str(fila["Direccion Domicilio"]).strip().upper()

    # Solo intentamos rescatar si hubo error vial
    if "NOT_VIAL_TYPE" in dir_norm:
        # Palabras clave que indican dirección rural válida
        keywords = [
            "KM",
            "KILOMETRO",
            "VIA",
            "VÍA",
            "VEREDA",
            "VDA",
            "LOTE",
            "FINCA",
            "CORREGIMIENTO",
            "SECTOR",
        ]

        # Si empieza con alguna de esas, la dejamos pasar limpia
        for k in keywords:
            if dir_orig.startswith(k):
                return dir_orig  # Devolvemos la original limpia

    return dir_norm


# Aplicamos el rescate
df["Direccion_Normalizada"] = df.apply(rescatar_rurales, axis=1)

# Verificamos cuánto mejoró
exito_nuevo = ~df["Direccion_Normalizada"].str.contains("Error", na=False)
print(f"Nuevo conteo de éxito: {exito_nuevo.sum()} de {len(df)}")

Nuevo conteo de éxito: 1908 de 2031


### 🏛️ 4. Enriquecimiento con Fuentes Oficiales (Cruce REPS)
Para mejorar la tasa de éxito, cruzamos nuestra base con el **Registro Especial de Prestadores de Servicios de Salud (REPS)** del Ministerio de Salud.

* **Estrategia:** Generamos "llaves de cruce" normalizadas (Prestador + Sede + Municipio) para buscar la dirección oficial reportada al gobierno si la nuestra está defectuosa.
* **Jerarquía de Decisión:**
    1.  Si nuestra dirección normalizada es buena $\rightarrow$ Se mantiene.
    2.  Si es mala, pero el REPS tiene una válida $\rightarrow$ Se adopta la del REPS.
    3.  Si ambas fallan $\rightarrow$ Se marca para investigación con IA.

In [ ]:
archivo = "Sedes.xls"

try:
    # Intenta leer el archivo con la codificación 'latin1'
    lista_de_dfs = pd.read_html(archivo, encoding="latin1", header=0)
    df_sedes = lista_de_dfs[0]

    print("¡Éxito! Archivo cargado con codificación 'latin1'.")
    df_sedes.head()

except Exception as e:
    print(f"Error con 'latin1': {e}")
    print("Prueba el Intento B.")

¡Éxito! Archivo cargado con codificación 'latin1'.


In [ ]:
# =============================================================================
# PASO 1: PREPARACIÓN Y NORMALIZACIÓN (Generación de columnas v10)
# =============================================================================
print("1. Aplicando Estandarizador v12 a ambas bases...")

# Aplicamos a tu base original
if "Direccion_Normalizada" not in df.columns:
    df["Direccion_Normalizada"] = df["Direccion Domicilio"].apply(
        estandarizador_maestro_v12
    )

# Aplicamos a la base del Ministerio (REPS)
# Asegúrate de que df_sedes esté cargado. Si no, descomenta la línea de abajo:
# df_sedes = pd.read_csv('sedes_reps.csv')
df_sedes["direccion_reps_v10"] = df_sedes["direccion"].apply(estandarizador_maestro_v12)

# =============================================================================
# PASO 2: CREACIÓN DE LLAVES Y MERGE (Aquí nace df_merged)
# =============================================================================
print("2. Generando llaves y cruzando bases...")


def limpiar_llave(texto):
    return unidecode(str(texto)).upper().strip() if pd.notna(texto) else ""


# Llaves en TU base (df)
df["Prestador_Key"] = df["Prestador"].apply(limpiar_llave)
df["Sucursal_Key"] = df["Sucursal Prestador"].apply(limpiar_llave)
df["Depto_Key"] = df["Departamento"].apply(limpiar_llave)
df["Muni_Key"] = df["Municipio"].apply(limpiar_llave)

# Llaves en base REPS (Ministerio)
df_sedes["prestador_clean"] = df_sedes["nombre_prestador"].apply(limpiar_llave)
df_sedes["nombre_clean"] = df_sedes["nombre"].apply(limpiar_llave)
df_sedes["depto_clean"] = df_sedes["departamento"].apply(limpiar_llave)
df_sedes["muni_clean"] = df_sedes["municipio"].apply(limpiar_llave)

# Deduplicar REPS (Para evitar multiplicar filas)
sedes_unicas = df_sedes.drop_duplicates(
    subset=["prestador_clean", "nombre_clean", "depto_clean", "muni_clean"],
    keep="first",
)

# EL CRUCE (MERGE) -> Aquí se crea la variable que te faltaba
df_merged = pd.merge(
    df,
    sedes_unicas[
        [
            "prestador_clean",
            "nombre_clean",
            "depto_clean",
            "muni_clean",
            "direccion_reps_v10",
        ]
    ],
    left_on=["Prestador_Key", "Sucursal_Key", "Depto_Key", "Muni_Key"],
    right_on=["prestador_clean", "nombre_clean", "depto_clean", "muni_clean"],
    how="left",
)

print(f"   > Cruce exitoso. Total registros en df_merged: {len(df_merged)}")

# =============================================================================
# PASO 3: SELECCIÓN CON PRIORIDAD (TU BASE > MINISTERIO > RURAL)
# =============================================================================


def seleccionar_con_prioridad_propia(fila):
    """
    Prioridad: 1. Tu Base (v10) -> 2. Ministerio (v10) -> 3. Rural -> 4. OpenAI
    """
    dir_tuya = str(fila.get("Direccion_Normalizada", ""))
    dir_reps = str(fila.get("direccion_reps_v10", ""))
    dir_cruda = str(fila.get("Direccion Domicilio", "")).upper()

    # Filtro de validez
    def es_valida(d):
        return (
            ("Error" not in d)
            and (d not in ["nan", "", "None", "NONE"])
            and (len(d) > 4)
        )

    # 1. TU DIRECCIÓN (PRIORIDAD)
    if es_valida(dir_tuya):
        return dir_tuya, "Original Normalizada", False

    # 2. MINISTERIO
    if es_valida(dir_reps):
        return dir_reps, "REPS (Ministerio)", False

    # 3. SALVAVIDAS RURAL
    keywords = ["KM ", "VIA ", "VEREDA", "VDA ", "CORREGIMIENTO", "LOTE ", "FINCA "]
    if any(k in dir_cruda for k in keywords):
        return dir_cruda, "Rescate Rural", False

    # 4. OPENAI
    return dir_tuya, "Error (Pendiente)", True


print("3. Clasificando direcciones...")
# Ahora sí funcionará porque df_merged ya existe
resultado = df_merged.apply(
    seleccionar_con_prioridad_propia, axis=1, result_type="expand"
)

df_merged["Direccion_Candidata"] = resultado[0]
df_merged["Fuente_Dato"] = resultado[1]
df_merged["Necesita_IA"] = resultado[2]

# =============================================================================
# PASO 4: RESULTADOS
# =============================================================================
df_resueltos = df_merged[~df_merged["Necesita_IA"]].copy()
df_para_openai = df_merged[df_merged["Necesita_IA"]].copy()

print("\n" + "=" * 40)
print("       BALANCE FINAL (df_merged)      ")
print("=" * 40)
print(f"Total Registros:              {len(df_merged)}")
print(
    f"✅ YA RESUELTOS:              {len(df_resueltos)} ({len(df_resueltos) / len(df_merged):.1%})"
)
print(
    f"🤖 PENDIENTES PARA OPENAI:    {len(df_para_openai)} ({len(df_para_openai) / len(df_merged):.1%})"
)
print("=" * 40)

if not df_para_openai.empty:
    print("\nEjemplos que van para IA:")
    print(df_para_openai[["Prestador", "Direccion_Normalizada"]].head())

1. Aplicando Estandarizador v12 a ambas bases...
2. Generando llaves y cruzando bases...
   > Cruce exitoso. Total registros en df_merged: 2031
3. Clasificando direcciones...

       BALANCE FINAL (df_merged)      
Total Registros:              2031
✅ YA RESUELTOS:              1899 (93.5%)
🤖 PENDIENTES PARA OPENAI:    132 (6.5%)

Ejemplos que van para IA:
                                        Prestador  \
13                      GALLEGO URIBE JUAN CARLOS   
50                 POMARES CASTILLA SHIRLEY PATRY   
124  FIDEL  VASQUEZ LUCIGNIANI -  VASQUEZ LUCIGNI   
175                            HUMANA VITAL S.A.S   
176                            HUMANA VITAL S.A.S   

             Direccion_Normalizada  
13      Error: Formato Desconocido  
50           Error: Vía sin número  
124  Error: No es dirección física  
175                   Error: Vacío  
176                   Error: Vacío  


### 🤖 5. Agente de Recuperación con Inteligencia Artificial
Para los registros "huérfanos" (sin dirección válida en origen ni en REPS), desplegamos un agente basado en **GPT-3.5/4**.

* **Función:** Investiga la ubicación física de sedes específicas basándose en el nombre del prestador y el municipio.
* **Output:** Devuelve la dirección exacta o confirma la inexistencia de la sede (ej: *"No existe sede en Sopó, la principal está en Bogotá"*), evitando falsos positivos en el mapa.

In [ ]:
# ==========================================
# 1. CONFIGURACIÓN
# ==========================================
# ¡PEGA AQUÍ TU NUEVA CLAVE GENERADA!
OPENAI_API_KEY = "opeaikey"

client = openai.OpenAI(api_key=OPENAI_API_KEY)


# ==========================================
# 2. FUNCIÓN DE INVESTIGACIÓN (INTACTA)
# ==========================================
def buscar_sucursal_especifica(fila):
    # Extraemos los datos con seguridad (evitando errores por nulos)
    prestador = str(fila.get("Prestador", "")).strip()
    sucursal = str(fila.get("Sucursal Prestador", "")).strip()
    municipio = str(fila.get("Municipio", "")).strip()
    depto = str(fila.get("Departamento", "")).strip()
    pista_dir = str(fila.get("Direccion Domicilio", "Pista no disponible")).strip()

    # Si la pista es muy corta o nula, le decimos "No disponible"
    if len(pista_dir) < 3 or "nan" in pista_dir.lower():
        pista_dir = "No disponible"

    # --- EL PROMPT (LAS INSTRUCCIONES) ---
    prompt = f"""
    Actúa como un experto en ubicación de sedes médicas en Colombia.
    
    OBJETIVO:
    Encontrar la dirección física EXACTA de una sede específica dentro del municipio indicado.
    
    DATOS:
    1. Entidad/Prestador: {prestador}
    2. Sede/Sucursal: {sucursal}
    3. Municipio: {municipio}
    4. Departamento: {depto}
    5. Pista de dirección: "{pista_dir}"
    
    REGLAS OBLIGATORIAS:
    - Si existe sede en el municipio indicado, devuelve ÚNICAMENTE la dirección física (ej: "Calle 10 #5-21").
    - Si NO existe sede en ese municipio:
    - NO inventes direcciones.
    - NO cambies de municipio.
    - NO uses frases como "Lo siento", "No tengo información" o disculpas.
    - Devuelve UNA frase breve que indique:
        1) que no existe sede en ese municipio,
        2) y el municipio donde realmente está la sede principal.
    EJEMPLOS:
    - "No existe sede en Villanueva; la sede principal está en Arjona."
    - "No existe sede en Bucarasica; la sede principal está en Tibú."
    
    FORMATO DE RESPUESTA:
    - Si existe sede → solo la dirección.
    - Si NO existe sede → solo la frase breve.
    
    """

    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",  # Usa "gpt-4" si quieres máxima precisión
            messages=[
                {
                    "role": "system",
                    "content": "Eres un sistema de georreferenciación preciso. Solo respondes direcciones.",
                },
                {"role": "user", "content": prompt},
            ],
            temperature=0.0,  # Cero creatividad, máxima precisión
            max_tokens=100,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"ERROR: {str(e)}"


# ==========================================
# 3. EJECUCIÓN CONECTADA AL CONSOLIDADO
# ==========================================

print("--- INICIANDO FASE DE INTELIGENCIA ARTIFICIAL ---")

# A. Usamos la bandera 'Necesita_IA' generada en el paso anterior
# Esto asegura que SOLO procesemos lo que quedó marcado como pendiente.
mask_faltantes = df_merged["Necesita_IA"] == True

cantidad_a_investigar = mask_faltantes.sum()
print(f"Investigando {cantidad_a_investigar} registros críticos con OpenAI...")

if cantidad_a_investigar > 0:
    # B. Aplicamos la función SOLO al subconjunto filtrado
    # Se guarda en una nueva columna 'Direccion_IA_Rescate'
    print("Consultando API (esto puede tardar unos segundos)...")
    direcciones_ia = df_merged.loc[mask_faltantes].apply(
        buscar_sucursal_especifica, axis=1
    )

    # C. Asignamos los resultados al DataFrame principal
    df_merged.loc[mask_faltantes, "Direccion_IA_Rescate"] = direcciones_ia

    # D. Mostramos muestra de resultados
    cols_ver = ["Prestador", "Municipio", "Direccion_Candidata", "Direccion_IA_Rescate"]
    print("\n--- RESULTADOS DE LA INVESTIGACIÓN IA ---")
    print(df_merged.loc[mask_faltantes, cols_ver].head(10))

else:
    print(
        "¡Excelente! No hay registros pendientes para IA. Todo se resolvió con normalización y REPS."
    )

# ==========================================
# 4. INTEGRACIÓN FINAL (CIERRE DEL PROCESO)
# ==========================================
# Si la IA trajo algo, lo ponemos como dirección candidata final
# Si la IA falló o dijo "No existe", se queda el error original.


def integrar_ia(fila):
    dir_actual = fila["Direccion_Candidata"]
    dir_ia = str(fila.get("Direccion_IA_Rescate", ""))

    # Si necesitábamos IA y la IA trajo algo que no sea error de API
    if fila["Necesita_IA"] and pd.notna(dir_ia) and "ERROR:" not in dir_ia:
        return dir_ia  # Nos quedamos con lo que dijo la IA

    return dir_actual


df_merged["Direccion_Definitiva_Final"] = df_merged.apply(integrar_ia, axis=1)

print(f"\nProceso finalizado. Columna final: 'Direccion_Definitiva_Final'")

--- INICIANDO FASE DE INTELIGENCIA ARTIFICIAL ---
Investigando 132 registros críticos con OpenAI...
Consultando API (esto puede tardar unos segundos)...

--- RESULTADOS DE LA INVESTIGACIÓN IA ---
                                             Prestador            Municipio  \
13                           GALLEGO URIBE JUAN CARLOS            MANIZALES   
50                      POMARES CASTILLA SHIRLEY PATRY  CARTAGENA DE INDIAS   
124       FIDEL  VASQUEZ LUCIGNIANI -  VASQUEZ LUCIGNI        FLORIDABLANCA   
175                                 HUMANA VITAL S.A.S               MADRID   
176                                 HUMANA VITAL S.A.S             MOSQUERA   
221                                  AEROSANIDAD S.A.S             RIONEGRO   
234  AYUDAS MEDICAS DOMICILIARIAS YEMPRESARIAL SOLU...                 COTA   
237                           SANTILHEMERGENCIAS S.A.S               CAJICA   
238                           SANTILHEMERGENCIAS S.A.S            ZIPAQUIRA   
239           

In [ ]:
df_merged["Direccion_IA_Rescate"].unique()

array([nan, '"CLIN LA PRESENTACION CONS 327"',
       'No existe sede en Cartagena de Indias; la sede principal está en Bolívar.',
       'No existe sede en Floridablanca; la sede principal está en Bucaramanga.',
       'No existe sede en Madrid; la sede principal está en Bogotá.',
       'No existe sede en Mosquera; la sede principal está en Bogotá.',
       'No existe sede en Rionegro; la sede principal está en Medellín.',
       'No existe sede en Cota; la sede principal está en Bogotá.',
       'No existe sede en Cajicá; la sede principal está en Bogotá.',
       'No existe sede en Zipaquirá; la sede principal está en Bogotá.',
       'No existe sede en Tocancipá; la sede principal está en Bogotá.',
       'No existe sede en COTA; la sede principal está en BOGOTÁ.',
       'No existe sede en Sopó; la sede principal está en Bogotá.',
       'No existe sede en Sabaneta; la sede principal está en Medellín.',
       'Carrera 50 # 51-50, Bello, Antioquia.',
       'No existe sede en Cal

In [ ]:
# =============================================================================
# 1. AJUSTE CRÍTICO: LISTA DE CORTE CORREGIDA (V11)
# =============================================================================
# ELIMINAMOS "B", "L", "CON" de la lista de corte porque son parte de direcciones reales
# (Ej: Carrera 7B, Localidad (L), Calle 5 con 4)

STOP_WORDS_LIST_V11 = [
    # Inmuebles
    "APARTAMENTO",
    "APTO",
    "APT",
    "CASA",
    "TORRE",
    "OFICINA",
    "OF",
    "CONSULTORIO",
    "CONS",
    "CONSU",
    "CS",  # QUITAMOS "CON"
    "BLOQUE",
    "BL",
    "UNIDAD",
    "INT",
    "INTERIOR",
    "PISO",
    "PIS",
    "PI",
    "PA",
    "P",
    "LOCAL",
    "LC",
    "ACCESO",
    "ACC",
    "ENTRADA",
    "HANGAR",
    "BODEGA",
    "AEROPUERTO",
    # Zonas (QUITAMOS "B" SOLA)
    "BARRIO",
    "BR",
    "BRR",
    "BRRIO",
    "URBANIZACION",
    "URB",
    "CONJUNTO",
    "CONJ",
    "ETAPA",
    "SECTOR",
    "SEC",
    "ZONA",
    "CENTRO COMERCIAL",
    "CC",
    "MALL",
    "PLAZA",
    "PARQUE",
    "COND",
    "CONDOMINIO",
    "VENEZIA",
    "BRISAS",
    "RIO",
    # Instituciones
    "AVENIDA",
    "AV",
    "EDIFICIO",
    "ED",
    "EDIF",
    "CLINICA",
    "CLIN",
    "HOSPITAL",
    "IPS",
    "ESE",
    "LINICA",
    "SOMA",
    "JASBAN",
    "PORTO",
    "PORTOAZ",
    "MEDICA",
    "VIVENZA",
    "NOVA",
    "NOVACENTRO",
    "INNOVO",
    "COLEGIO",
    "U",
    "UNV",
    "UNIVERSIDAD",
    "CORREDOR",
    "UMBRI",
    "COMUNCA",
    # Ciudades y Relleno
    "ENVIGADO",
    "CHIA",
    "CAJICA",
    "BUCARAMANGA",
    "MEDELLIN",
    "CALI",
    "BOGOTA",
    "CARTAGENA",
    "BARRANQUILLA",
    "SOACHA",
    "BELLO",
    "ITAGUI",
    "PEREIRA",
    "MANIZALES",
    "EL",
    "LA",
    "LOS",
    "LAS",
    "DEL",
    "DE",
    "FRENTE",
    "DIAGONAL A",
    "AL LADO",
    "ESQUINA",
    "ESQ",
    "COMUNA",
    "COM",
    "LOCALIDAD",
    "LOC",
    "CASCO",
    "URBANO",
    "PUERTA",
]

STOP_WORDS_RE_V11 = re.compile(
    r"\b(" + "|".join(STOP_WORDS_LIST_V11) + r")\b", flags=re.IGNORECASE
)


def estandarizador_maestro_v11(direccion):
    if pd.isna(direccion) or str(direccion).strip() == "":
        return "Error: Vacío"

    # 1. Limpieza inicial de la respuesta IA (Quitar comillas y puntos finales)
    text_raw = str(direccion).strip().replace('"', "").replace("'", "").rstrip(".")

    # 2. Estrategia de la Coma (La IA suele poner: Dirección, Ciudad)
    # Cortamos todo lo que esté después de la primera coma
    if "," in text_raw:
        text_raw = text_raw.split(",")[0]

    # 3. Pre-limpieza técnica
    text_raw = (
        text_raw.upper()
        .replace("N°", " ")
        .replace("Nº", " ")
        .replace("°", " ")
        .replace("º", " ")
    )
    text = unidecode(text_raw).strip()
    text = (
        text.replace("NO ", " ")
        .replace(" N ", " ")
        .replace("#", " ")
        .replace("-", " ")
        .replace(".", " ")
    )

    # Separar letras y números (KR7B -> KR 7 B)
    text = re.sub(r"([A-Z]+)(\d+)", r"\1 \2", text)
    text = re.sub(r"(\d+)([A-Z]+)", r"\1 \2", text)

    mapeo = {
        "CALLE": "CL",
        "CLL": "CL",
        "CL": "CL",
        "C": "CL",
        "CARRERA": "KR",
        "CRA": "KR",
        "KRA": "KR",
        "CR": "KR",
        "KR": "KR",
        "K": "KR",
        "DIAGONAL": "DG",
        "DIAG": "DG",
        "DG": "DG",
        "TRANSVERSAL": "TV",
        "TRANS": "TV",
        "TR": "TV",
        "TV": "TV",
        "AVENIDA": "AV",
        "AV": "AV",
        "AVE": "AV",
        "CIRCULAR": "CQ",
        "VIA": "VIA",
        "KM": "KM",
        "VEREDA": "VDA",
    }

    text = re.sub(r"\s+", " ", text).strip()
    parts = text.split()
    if not parts:
        return "Error: Sin info"

    primer = parts[0]

    # LÓGICA DE CORTE SUAVE (Solo corta si encuentra palabras de la lista v11)
    if primer in mapeo:
        tipo = mapeo[primer]
        resto = " ".join(parts[1:])

        match = STOP_WORDS_RE_V11.search(resto)
        if match:
            resto = resto[: match.start()].strip()

        return f"{tipo} {resto}".strip()

    # Si la IA devolvió "CR 7..." (y CR no estaba mapeado en el loop anterior)
    # Intentamos mapear de nuevo todo el string
    if primer in ["CR", "CRA", "K", "C"]:
        # Mapeo manual rápido
        if primer in ["CR", "CRA", "K"]:
            return estandarizador_maestro_v11("KR " + " ".join(parts[1:]))
        if primer == "C":
            return estandarizador_maestro_v11("CL " + " ".join(parts[1:]))

    # Si empieza con número, devolvemos error (o el texto si confiamos en la IA)
    if primer.isdigit():
        return "Error: Formato Numérico"

    return text  # Devolvemos lo que dijo la IA limpio si no pudimos estandarizar


# =============================================================================
# 2. CONEXIÓN Y UNIFICACIÓN FINAL
# =============================================================================

print("Procesando direcciones recuperadas por Inteligencia Artificial...")


# Función para decidir qué dirección final usar
def consolidar_final(fila):
    # 1. Si ya teníamos una dirección buena (Original o REPS), la dejamos quieta
    # (Usamos la columna que ya habías calculado antes)
    if not fila["Necesita_IA"]:
        return fila["Direccion_Candidata"]

    # 2. Si necesitábamos IA, procesamos lo que trajo
    texto_ia = str(fila.get("Direccion_IA_Rescate", ""))

    # Filtros de fallo de IA
    if (
        pd.isna(texto_ia)
        or "ERROR" in texto_ia
        or "No existe" in texto_ia
        or "sede principal" in texto_ia
    ):
        return "NO ENCONTRADO"  # Se rinde

    # 3. Limpiamos la respuesta de la IA con la V11
    return estandarizador_maestro_v11(texto_ia)


# APLICAMOS LA CONSOLIDACIÓN
df_merged["Direccion_Definitiva_Final"] = df_merged.apply(consolidar_final, axis=1)

# =============================================================================
# 3. VERIFICACIÓN DE TUS CASOS ESPECÍFICOS
# =============================================================================
print("\n--- Verificando corrección de pérdida de datos ---")
casos_ver = ["Carrera 7B #12-106", "CRA 7B # 11C-21", "Cr. 7 B bis # 132 31"]
print(f"{'IA Original':30} | {'Antes (Errado)':15} | {'Ahora (Correcto)':15}")
for c in casos_ver:
    # Simulamos el error antiguo vs el nuevo
    # El error antiguo cortaba en la B
    print(f"{c:30} | ... KR 7 ...      | {estandarizador_maestro_v11(c)}")

print("\n--- Muestra Final del DataFrame ---")
mask_ia_exito = (df_merged["Necesita_IA"]) & (
    df_merged["Direccion_Definitiva_Final"] != "NO ENCONTRADO"
)
if mask_ia_exito.sum() > 0:
    cols = ["Direccion_IA_Rescate", "Direccion_Definitiva_Final"]
    print(df_merged.loc[mask_ia_exito, cols].head(10))

Procesando direcciones recuperadas por Inteligencia Artificial...

--- Verificando corrección de pérdida de datos ---
IA Original                    | Antes (Errado)  | Ahora (Correcto)
Carrera 7B #12-106             | ... KR 7 ...      | KR 7 B 12 106
CRA 7B # 11C-21                | ... KR 7 ...      | KR 7 B 11 C 21
Cr. 7 B bis # 132 31           | ... KR 7 ...      | KR 7 B BIS 132 31

--- Muestra Final del DataFrame ---
                                  Direccion_IA_Rescate  \
13                     "CLIN LA PRESENTACION CONS 327"   
256              Carrera 50 # 51-50, Bello, Antioquia.   
266  Carrera 29 # 31-30, El Carmen de Viboral, Anti...   
268                       Carrera 29 # 29-10, Palmira.   
441                        Carrera 7 # 116-21, Bogotá.   
494                               "Carrera 53 #31-141"   
617                     CL ANZOATEGUI, Tumaco, Nariño.   
699  "CENTRO DE SALUD LA NOHORA se encuentra en SEC...   
700           Carrera 33 # 40-01, Villavicencio, 

In [ ]:
# Verificar prestador
filtro = df_merged["Prestador"] == "ALEJANDRA DEL PILAR GARCIÁ GALLO"
a = df_merged[filtro]
a

,Tipo de Documento,Documento Prestador,Prestador,Sucursal Prestador,Nivel Complejidad,Oficina Sucursal,Departamento,Cod. Departamento,Municipio,Cod. Municipio,...,prestador_clean,nombre_clean,depto_clean,muni_clean,direccion_reps_v10,Direccion_Candidata,Fuente_Dato,Necesita_IA,Direccion_IA_Rescate,Direccion_Definitiva_Final
22,C,1093219037,ALEJANDRA DEL PILAR GARCIÁ GALLO,Alejandra del Pilar Garciá Gallo,2.0,OFICINA PEREIRA,RISARALDA,66,SANTA ROSA DE CABAL,68200,...,ALEJANDRA DEL PILAR GARCIA GALLO,ALEJANDRA DEL PILAR GARCIA GALLO,RISARALDA,SANTA ROSA DE CABAL,KR 7 B 12 106 PRIMER,KR 7 B 12 106,Original Normalizada,False,NaN,KR 7 B 12 106


### 🛡️ 6. Auditoría de Calidad y Scoring Final
Antes de georreferenciar, cada dirección resultante pasa por un **Auditor de Calidad** que asigna un puntaje (0-100).

* **Score 100 (Excelente):** Urbana estándar (`KR 10 5 20`).
* **Score 80 (Buena):** Rural o Hito (`KM 5 VIA CHIA`).
* **Score 40 (Regular):** Texto genérico pero ubicable.
* **Score 0 (Crítico):** Errores, vacíos o confirmaciones de "No existe" por parte de la IA.

> **Nota:** Solo los registros con `Score >= 40` pasarán a la fase de mapa.

In [ ]:
# =============================================================================
# 1. FUNCIÓN DE AUDITORÍA Y PUNTAJE (SCORE)
# =============================================================================
def auditar_calidad_final(fila):
    """
    Analiza la dirección final y retorna una tupla: (Puntaje, Categoría)
    """
    texto = str(fila.get("Direccion_Definitiva_Final", ""))
    t = texto.upper().strip()

    # --- NIVEL 0: ERRORES Y FALLOS DE IA ---
    # Detectamos patrones de fallo específicos (incluyendo lo que devuelve OpenAI cuando no encuentra nada)
    patrones_error = [
        "ERROR",
        "NAN",
        "NONE",
        "NO ENCONTRADO",
        "VACIO",
        "NO EXISTE",
        "SEDE PRINCIPAL ESTÁ EN",
        "NO DISPONIBLE",  # Fallos típicos de IA
        "ERROR DE API",
    ]

    if pd.isna(texto) or t == "" or any(p in t for p in patrones_error):
        return 0, "CRÍTICO (No Ubicable)"

    # --- NIVEL 100: DIRECCIÓN URBANA ESTÁNDAR (ALTA PRECISIÓN) ---
    # Empieza con vía conocida y tiene números
    vias_urbanas = [
        "CL",
        "KR",
        "CR",
        "CRA",
        "AV",
        "DG",
        "TV",
        "CQ",
        "CARRERA",
        "CALLE",
        "CIRCULAR",
        "DIAGONAL",
        "TRANSVERSAL",
    ]
    # Verificamos que empiece con una vía Y tenga al menos un dígito
    if any(t.startswith(v + " ") for v in vias_urbanas) and any(
        char.isdigit() for char in t
    ):
        return 100, "EXCELENTE (Urbana Estándar)"

    # --- NIVEL 80: DIRECCIÓN RURAL O LUGAR (BUENA PRECISIÓN) ---
    # Contiene palabras clave de ubicación rural o hitos
    rurales = [
        "VIA",
        "KM",
        "VEREDA",
        "VDA",
        "CORREGIMIENTO",
        "LOTE",
        "FINCA",
        "HACIENDA",
        "MANZANA",
        "MZ",
    ]
    if any(r in t for r in rurales):
        return 80, "BUENA (Rural/Hito)"

    # --- NIVEL 40: TEXTO GENÉRICO (BAJA PRECISIÓN) ---
    # No es error, pero tampoco es una dirección estándar (ej: "CENTRO MEDICO X")
    # Sirve para intentar ubicar, pero con desconfianza.
    if len(t) > 4:
        return 40, "REGULAR (Nombre/Texto)"

    # Si no cae en nada anterior
    return 0, "CRÍTICO (Formato Inválido)"


# =============================================================================
# 2. APLICACIÓN DE LA AUDITORÍA
# =============================================================================
print("Calculando Score de Calidad para las direcciones finales...")

# Aplicamos la función y separamos en dos columnas
resultado_auditoria = df_merged.apply(
    auditar_calidad_final, axis=1, result_type="expand"
)
df_merged["Score_Geo"] = resultado_auditoria[0]
df_merged["Calidad_Dato"] = resultado_auditoria[1]

# =============================================================================
# 3. REPORTE DE CALIDAD DE DATOS
# =============================================================================
conteo_calidad = df_merged["Calidad_Dato"].value_counts()
promedio_score = df_merged["Score_Geo"].mean()

print("\n" + "=" * 40)
print("       AUDITORÍA FINAL DE DIRECCIONES      ")
print("=" * 40)
print(f"Puntaje Promedio de la Base: {promedio_score:.1f} / 100")
print("-" * 40)
print("DISTRIBUCIÓN DE CALIDAD:")
print(conteo_calidad)
print("=" * 40)

# =============================================================================
# 4. LIMPIEZA DE DUPLICADOS INTELIGENTE
# =============================================================================
# Si tienes filas repetidas (Mismo Prestador, Ciudad y Servicio),
# nos quedamos con la que tenga MEJOR Score.

print("\nEliminando duplicados priorizando la mejor calidad de dirección...")

# 1. Definimos qué hace única a una oferta de servicio
cols_clave_servicio = [
    "Prestador",
    "Sucursal Prestador",
    "Departamento",
    "Municipio",
    "Concepto Factura",  # El servicio que venden
]

# 2. Ordenamos: Primero por las claves, luego por Score descendente (Mejor arriba)
df_merged.sort_values(
    by=cols_clave_servicio + ["Score_Geo"],
    ascending=[True, True, True, True, True, False],
    inplace=True,
)

# 3. Borramos duplicados quedándonos con el primero (el de mejor score)
len_antes = len(df_merged)
df_final_unico = df_merged.drop_duplicates(
    subset=cols_clave_servicio, keep="first"
).copy()
len_despues = len(df_final_unico)

print(f"Registros iniciales: {len_antes}")
print(f"Registros finales:   {len_despues}")
print(f"Duplicados eliminados: {len_antes - len_despues}")

# Exportar resultado final para mapa
# df_final_unico.to_excel("Base_Consolidada_Para_Mapa.xlsx", index=False)

Calculando Score de Calidad para las direcciones finales...

       AUDITORÍA FINAL DE DIRECCIONES      
Puntaje Promedio de la Base: 93.3 / 100
----------------------------------------
DISTRIBUCIÓN DE CALIDAD:
Calidad_Dato
EXCELENTE (Urbana Estándar)    1857
CRÍTICO (No Ubicable)           111
BUENA (Rural/Hito)               38
REGULAR (Nombre/Texto)           21
CRÍTICO (Formato Inválido)        4
Name: count, dtype: int64

Eliminando duplicados priorizando la mejor calidad de dirección...
Registros iniciales: 2031
Registros finales:   2031
Duplicados eliminados: 0


In [ ]:
# Filtramos y mostramos para ver qué son exactamente
cols_ver = [
    "Prestador",
    "Municipio",
    "Direccion Domicilio",
    "Direccion_Definitiva_Final",
    "Calidad_Dato",
]

print("--- INSPECCIÓN DE LOS 4 INVÁLIDOS ---")
# Estos suelen ser basura como "1", "A", "SUR"
df_inv = df_merged[df_merged["Calidad_Dato"] == "CRÍTICO (Formato Inválido)"]
print(df_inv[cols_ver])

print("\n" + "=" * 50 + "\n")

print("--- INSPECCIÓN DE MUESTRA NO UBICABLES (Top 10) ---")
# Estos deberían ser los "No existe sede en..."
df_no_ubic = df_merged[df_merged["Calidad_Dato"] == "CRÍTICO (No Ubicable)"]
print(df_no_ubic[cols_ver].head(10))

--- INSPECCIÓN DE LOS 4 INVÁLIDOS ---
                                            Prestador    Municipio  \
755   CENTRO DE SALUD DE LA SALINA-RED SALUD CASANARE    LA SALINA   
1566                            CLINICA DEL OCCIDENTE       BOGOTA   
911                E.S.E HOSPITAL SAN RAFAEL DE ANDES        ANDES   
1869                                      MUTALIS SAS  SANTA MARTA   

                 Direccion Domicilio Direccion_Definitiva_Final  \
755   Diagonal a la estación de poli                       DG A   
1566        AV LAS AMERICAS  71 C-29                         AV   
911               AV MEDELLIN  48-20                         AV   
1869       AV del Ferrocarril 29-200                         AV   

                    Calidad_Dato  
755   CRÍTICO (Formato Inválido)  
1566  CRÍTICO (Formato Inválido)  
911   CRÍTICO (Formato Inválido)  
1869  CRÍTICO (Formato Inválido)  


--- INSPECCIÓN DE MUESTRA NO UBICABLES (Top 10) ---
              Prestador                Municip

In [ ]:
# =============================================================================
# 1. PARCHE QUIRÚRGICO (Salvar a los 4 soldados caídos)
# =============================================================================
print("Aplicando correcciones manuales a direcciones específicas...")

# Caso 1: AV LAS AMERICAS (Bogotá) - Se cortó por 'LAS'
mask_americas = df_merged["Direccion Domicilio"].str.contains(
    "AV LAS AMERICAS", case=False, na=False
)
df_merged.loc[mask_americas, "Direccion_Definitiva_Final"] = "AV AMERICAS 71 C 29"
df_merged.loc[mask_americas, "Score_Geo"] = 100  # Lo volvemos excelente manualmente

# Caso 2: AV MEDELLIN (Andes) - Se cortó por 'MEDELLIN'
mask_medellin = df_merged["Direccion Domicilio"].str.contains(
    "AV MEDELLIN", case=False, na=False
)
df_merged.loc[mask_medellin, "Direccion_Definitiva_Final"] = "AV MEDELLIN 48 20"
df_merged.loc[mask_medellin, "Score_Geo"] = 100

# Caso 3: AV DEL FERROCARRIL (Santa Marta) - Se cortó por 'DEL'
mask_ferro = df_merged["Direccion Domicilio"].str.contains(
    "AV del Ferrocarril", case=False, na=False
)
df_merged.loc[mask_ferro, "Direccion_Definitiva_Final"] = "AV FERROCARRIL 29 200"
df_merged.loc[mask_ferro, "Score_Geo"] = 100

# Caso 4: LA SALINA (Dirección descriptiva)
# Como no tiene nomenclatura, le ponemos "CENTRO" para que ArcGIS nos lleve al pueblo
mask_salina = df_merged["Direccion Domicilio"].str.contains(
    "Diagonal a la estación", case=False, na=False
)
df_merged.loc[mask_salina, "Direccion_Definitiva_Final"] = "CENTRO"
df_merged.loc[mask_salina, "Score_Geo"] = 50  # Aceptable para pueblo pequeño

print("Correcciones aplicadas.")

Aplicando correcciones manuales a direcciones específicas...
Correcciones aplicadas.


### 📍 7. Georreferenciación de Alta Precisión (ArcGIS)
Con la data limpia y validada, conectamos con el motor de geocodificación de **ArcGIS**.

* **Proceso:** Se envía la cadena jerárquica: `Dirección + Municipio + Departamento + Colombia`.
* **Resultado:** Obtenemos Latitud y Longitud precisas para ubicar al prestador en el mapa interactivo.

In [ ]:
# =============================================================================
# 2. LIMPIEZA FINAL Y FILTRO DE CALIDAD
# =============================================================================

# Definimos unicidad por Prestador + Sede + Ciudad + Servicio
cols_clave = [
    "Prestador",
    "Sucursal Prestador",
    "Departamento",
    "Municipio",
    "Concepto Factura",
]

# Ordenamos para que, si hay duplicados, quede arriba el que tiene mejor Score
df_merged.sort_values(
    by=cols_clave + ["Score_Geo"],
    ascending=[True, True, True, True, True, False],
    inplace=True,
)

# Eliminamos duplicados
df_final_unico = df_merged.drop_duplicates(subset=cols_clave, keep="first").copy()

# FILTRO DE ORO: Solo Score >= 40
# Aquí se eliminan automáticamente:
# 1. Los 113 vacíos de "No existe sede".
# 2. El caso de "La Salina" (que tiene Score 0).
# 3. Cualquier otro error crítico.
df_mapa = df_final_unico[df_final_unico["Score_Geo"] >= 40].copy()

print(f"\nRegistros totales únicos: {len(df_final_unico)}")
print(f"Registros APTOS para mapa: {len(df_mapa)}")
print(f"Basura eliminada: {len(df_final_unico) - len(df_mapa)}")

# =============================================================================
# 3. GEOREFERENCIACIÓN (ARCGIS)
# =============================================================================
geolocator = ArcGIS(timeout=10)


def obtener_coordenadas(fila):
    # Armamos query robusto
    query = f"{fila['Direccion_Definitiva_Final']}, {fila['Municipio']}, {fila['Departamento']}, Colombia"
    try:
        location = geolocator.geocode(query)
        if location:
            return location.latitude, location.longitude
        return None, None
    except:
        time.sleep(1)  # Reintento de seguridad
        try:
            location = geolocator.geocode(query)
            if location:
                return location.latitude, location.longitude
        except:
            pass
        return None, None


print("\n--- INICIANDO GEOREFERENCIACIÓN ---")
print("Consultando ArcGIS... (Esto tomará unos minutos)")

resultados = []
total_mapa = len(df_mapa)

for i, (idx, row) in enumerate(df_mapa.iterrows()):
    lat, lon = obtener_coordenadas(row)
    resultados.append({"Index": idx, "Latitud": lat, "Longitud": lon})

    if (i + 1) % 50 == 0:
        print(f"Progreso: {i + 1}/{total_mapa}...")

# Unimos coordenadas al dataframe
df_coords = pd.DataFrame(resultados).set_index("Index")
df_mapa = df_mapa.join(df_coords)

# =============================================================================
# 4. EXPORTACIÓN FINAL
# =============================================================================
# Filtramos los que ArcGIS no encontró (si hubo alguno)
df_mapa_final = df_mapa[df_mapa["Latitud"].notna()].copy()

print("\n" + "=" * 40)
print(f"PROCESO FINALIZADO")
print(f"✅ Direcciones georreferenciadas: {len(df_mapa_final)}")
print(f"❌ Fallos de ArcGIS: {len(df_mapa) - len(df_mapa_final)}")
print("=" * 40)

archivo_salida = "Base_Prestadores_Final_Para_Streamlit.xlsx"
# df_mapa_final.to_excel(archivo_salida, index=False)
# print(f"Archivo guardado: {archivo_salida}")


Registros totales únicos: 2031
Registros APTOS para mapa: 1920
Basura eliminada: 111

--- INICIANDO GEOREFERENCIACIÓN ---
Consultando ArcGIS... (Esto tomará unos minutos)
Progreso: 50/1920...
Progreso: 100/1920...
Progreso: 150/1920...
Progreso: 200/1920...
Progreso: 250/1920...
Progreso: 300/1920...
Progreso: 350/1920...
Progreso: 400/1920...
Progreso: 450/1920...
Progreso: 500/1920...
Progreso: 550/1920...
Progreso: 600/1920...
Progreso: 650/1920...
Progreso: 700/1920...
Progreso: 750/1920...
Progreso: 800/1920...
Progreso: 850/1920...
Progreso: 900/1920...
Progreso: 950/1920...
Progreso: 1000/1920...
Progreso: 1050/1920...
Progreso: 1100/1920...
Progreso: 1150/1920...
Progreso: 1200/1920...
Progreso: 1250/1920...
Progreso: 1300/1920...
Progreso: 1350/1920...
Progreso: 1400/1920...
Progreso: 1450/1920...
Progreso: 1500/1920...
Progreso: 1550/1920...
Progreso: 1600/1920...
Progreso: 1650/1920...
Progreso: 1700/1920...
Progreso: 1750/1920...
Progreso: 1800/1920...
Progreso: 1850/1920.

In [ ]:
# =============================================================================
# CORRECCIÓN DE ETIQUETAS (ASIGNAR CRÉDITO A LA IA)
# =============================================================================

# Lógica: Si estaba marcado como "Necesita_IA" y logramos un Score alto (entró al mapa),
# entonces fue un ÉXITO de la IA.

mask_exito_ia = (df_mapa["Necesita_IA"] == True) & (df_mapa["Score_Geo"] >= 40)

# Actualizamos la etiqueta solo para esos registros
df_mapa.loc[mask_exito_ia, "Fuente_Dato"] = "Agente IA (OpenAI)"

# =============================================================================
# RE-GENERAR LAS MÉTRICAS FINALES
# =============================================================================
print("--- MÉTRICAS FINALES CORREGIDAS ---")
total_final = len(df_mapa)

print(f"1. Total de Prestadores Georreferenciados: {total_final}")

# Calidad
print("\n2. Distribución por Calidad:")
print(df_mapa["Calidad_Dato"].value_counts())

# Fuentes (Ahora sí debe salir la IA)
print("\n3. Origen de los Datos (Trazabilidad Real):")
print(df_mapa["Fuente_Dato"].value_counts())

# Tipología
urbanas = (
    df_mapa["Direccion_Definitiva_Final"]
    .str.split()
    .str[0]
    .isin(["CL", "KR", "DG", "TV", "AV", "CQ"])
    .sum()
)
rurales = total_final - urbanas
print(f"\n4. Tipología Espacial:")
print(f"   - Urbanas: {urbanas} ({urbanas / total_final:.1%})")
print(f"   - Rurales: {rurales} ({rurales / total_final:.1%})")

--- MÉTRICAS FINALES CORREGIDAS ---
1. Total de Prestadores Georreferenciados: 1920

2. Distribución por Calidad:
Calidad_Dato
EXCELENTE (Urbana Estándar)    1857
BUENA (Rural/Hito)               38
REGULAR (Nombre/Texto)           21
CRÍTICO (Formato Inválido)        4
Name: count, dtype: int64

3. Origen de los Datos (Trazabilidad Real):
Fuente_Dato
Original Normalizada    1896
Agente IA (OpenAI)        21
REPS (Ministerio)          2
Rescate Rural              1
Name: count, dtype: int64

4. Tipología Espacial:
   - Urbanas: 1867 (97.2%)
   - Rurales: 53 (2.8%)


### 📊 8. Validación de Resultados y Exportación
Generamos el balance final del proyecto para medir el impacto de la limpieza.
* **KPIs:** Tasa de éxito (% ubicados), distribución por calidad y trazabilidad de la fuente (cuántos rescató la IA vs. Originales).
* **Output:** Generación del archivo plano final (`Base_Estructura_Final_Lista.xlsx`) con la estructura requerida para la carga en la aplicación Streamlit.

In [ ]:
# =============================================================================
# 1. CÁLCULO DE DISTANCIA Y CATEGORIZACIÓN (CORREGIDO)
# =============================================================================
def calcular_distancia(row):
    """Calcula la distancia geodésica en kilómetros entre la coordenada original y la nueva."""

    # 1. Chequeo por valores 0 o NaN
    if (
        pd.isna(row["Latitud"])
        or pd.isna(row["Valor Latitud"])
        or row["Valor Latitud"] == 0
        or row["Latitud"] == 0
    ):
        return np.nan

    # 2. Coordenadas: (Latitud, Longitud)
    # NOTA: Usamos 'Latitud_Final' y 'Longitud_Final' que son las coordenadas de ArcGIS.
    punto_orig = (row["Valor Latitud"], row["Valor Longitud"])
    punto_nuevo = (row["Latitud"], row["Longitud"])

    # 3. Cálculo de distancia
    return geodesic(
        punto_orig, punto_nuevo
    ).km  # Usando la función importada directamente


def categorizar_diferencia(distancia_km):
    """Categoriza la diferencia en base a umbrales de distancia."""
    if pd.isna(distancia_km):
        return "N/A (Faltante/Error Origen)"
    elif distancia_km < 0.1:  # Menos de 100 metros
        return "A. Excelente (< 100m)"
    elif distancia_km < 1.0:  # Entre 100m y 1km
        return "B. Buena (0.1 - 1 km)"
    elif distancia_km < 5.0:  # Entre 1km y 5km
        return "C. Aceptable (1 - 5 km)"
    else:  # Más de 5km
        return "D. Crítica (> 5 km)"


# =============================================================================
# 2. APLICACIÓN Y CONSOLIDACIÓN
# =============================================================================
print("Aplicando cálculo de distancia y categorización...")

# Aplicar cálculo y categorización (Asumiendo df_mapa_final existe)
df_mapa_final["Diferencia_KM"] = df_mapa_final.apply(calcular_distancia, axis=1)
df_mapa_final["Categoria_Diferencia"] = df_mapa_final["Diferencia_KM"].apply(
    categorizar_diferencia
)

# 3. Consolidar resultados en una tabla resumen
resumen_distancia = (
    df_mapa_final.groupby("Categoria_Diferencia", dropna=False)["Diferencia_KM"]
    .agg(Conteo=("count"), Promedio_KM=("mean"), Max_KM=("max"))
    .reset_index()
)

# Formatear la tabla
resumen_distancia["Promedio_KM"] = resumen_distancia["Promedio_KM"].round(3)
resumen_distancia["Max_KM"] = resumen_distancia["Max_KM"].round(3)

# Ordenar lógicamente la salida
orden_categorias = [
    "A. Excelente (< 100m)",
    "B. Buena (0.1 - 1 km)",
    "C. Aceptable (1 - 5 km)",
    "D. Crítica (> 5 km)",
    "N/A (Faltante/Error Origen)",
]
resumen_distancia["Categoria_Diferencia"] = pd.Categorical(
    resumen_distancia["Categoria_Diferencia"], categories=orden_categorias, ordered=True
)
resumen_distancia.sort_values("Categoria_Diferencia", inplace=True)

print("--- BALANCE GENERAL DE LA MEJORA GEOSPATIAL ---")
print("Comparación: Coordenadas Originales vs. Coordenadas ArcGIS")
print(
    resumen_distancia[
        ["Categoria_Diferencia", "Conteo", "Promedio_KM", "Max_KM"]
    ].to_string()
)

Aplicando cálculo de distancia y categorización...
--- BALANCE GENERAL DE LA MEJORA GEOSPATIAL ---
Comparación: Coordenadas Originales vs. Coordenadas ArcGIS
          Categoria_Diferencia  Conteo  Promedio_KM    Max_KM
0        A. Excelente (< 100m)     771        0.030     0.100
1        B. Buena (0.1 - 1 km)     395        0.378     0.998
2      C. Aceptable (1 - 5 km)     199        2.497     4.996
3          D. Crítica (> 5 km)     193       63.184  1161.512
4  N/A (Faltante/Error Origen)       0          NaN       NaN


In [ ]:
# Crear la copia de seguridad (Deep Copy)
df_urg = df_mapa_final.copy()

In [ ]:
# Lista EXACTA del orden de columnas deseado (La estructura original del archivo de entrada)
columnas_finales = [
    "Tipo de Documento",
    "Documento Prestador",
    "Prestador",
    "Sucursal Prestador",
    "Nivel Complejidad",
    "Oficina Sucursal",
    "Departamento",
    "Cod. Departamento",
    "Municipio",
    "Cod. Municipio",
    "Depatamento/Municipio",
    "Direccion Domicilio",
    "Especialidad Prestador",
    "Tipo Prestador",
    "Responsable Negociacion",
    "Tipo de Red",
    "Valor Latitud",
    "Valor Longitud",
    "Tarifa",
    "Código Servicio",
    "Concepto Factura",
    "Direccionamiento",
    "(S/N) Riesgo Biologico",
    "Autoatencion",
    "e-mail Principal",
    "Horario Habil",
    "Horario No Habil",
    "IVR",
    "Telefono",
    "Telefono Celular",
    "Telefono Fijo",
    "Tipo Utilizacion",
    "(S/N)  Entrega Medicamento Ambulatorio",  # Nota: El espacio extra puede ser crítico
    "(S/N) Permite Osteosintesis",
    "servicios_urg_prio",
]

# 1. Sustituimos los valores originales con las columnas limpias y nuevas
# La columna 'Direccion Domicilio' se actualiza con la versión limpia.
df_urg["Direccion Domicilio"] = df_urg["Direccion_Definitiva_Final"]
# Las coordenadas originales se actualizan con las coordenadas georreferenciadas.
df_urg["Valor Latitud"] = df_urg["Latitud"]
df_urg["Valor Longitud"] = df_urg["Longitud"]

# 2. Manejo de columnas faltantes
# Identificamos qué columnas del orden final están presentes en nuestro df_final_estructurado
columnas_presentes = [col for col in columnas_finales if col in df_urg.columns]

# 3. Filtramos / reordenamos
df_urg = df_urg[columnas_presentes]

print(f"Estructura final con {len(df_urg.columns)} columnas lista.")
print("Las primeras 5 filas son:")
print(df_urg.head().to_string())

# Guardar la estructura final lista para subir
df_urg.to_excel("Base_Estructura_Final_Lista.xlsx", index=False)

Estructura final con 34 columnas lista.
Las primeras 5 filas son:
     Tipo de Documento  Documento Prestador                                       Prestador                                  Sucursal Prestador  Nivel Complejidad Oficina Sucursal  Departamento  Cod. Departamento  Municipio  Cod. Municipio  Depatamento/Municipio             Direccion Domicilio Especialidad Prestador Tipo Prestador Responsable Negociacion    Tipo de Red  Valor Latitud  Valor Longitud    Tarifa  Código Servicio                    Concepto Factura  Direccionamiento (S/N) Riesgo Biologico Autoatencion                     e-mail Principal                        Horario Habil Horario No Habil IVR    Telefono Telefono Celular            Telefono Fijo Tipo Utilizacion (S/N)  Entrega Medicamento Ambulatorio (S/N) Permite Osteosintesis
679                  N            813007875   HOSPITALMUNICIPAL NUESTRA SEÑORADE GUADALUPE-  ESE HOSPITAL MUNICIPAL NUESTRA SEÑORA DE GUADALUPE                1.0   OFICINA BOGOTA  

In [41]:
print(df_urg.columns.tolist())

['Tipo de Documento', 'Documento Prestador', 'Prestador', 'Sucursal Prestador', 'Nivel Complejidad', 'Oficina Sucursal', 'Departamento', 'Cod. Departamento', 'Municipio', 'Cod. Municipio', 'Depatamento/Municipio', 'Direccion Domicilio', 'Especialidad Prestador', 'Tipo Prestador', 'Responsable Negociacion', 'Tipo de Red', 'Valor Latitud', 'Valor Longitud', 'Tarifa', 'Código Servicio', 'Concepto Factura', 'Direccionamiento', '(S/N) Riesgo Biologico', 'Autoatencion', 'e-mail Principal', 'Horario Habil', 'Horario No Habil', 'IVR', 'Telefono', 'Telefono Celular', 'Telefono Fijo', 'Tipo Utilizacion', '(S/N)  Entrega Medicamento Ambulatorio', '(S/N) Permite Osteosintesis']


In [42]:
df_urg

,Tipo de Documento,Documento Prestador,Prestador,Sucursal Prestador,Nivel Complejidad,Oficina Sucursal,Departamento,Cod. Departamento,Municipio,Cod. Municipio,...,e-mail Principal,Horario Habil,Horario No Habil,IVR,Telefono,Telefono Celular,Telefono Fijo,Tipo Utilizacion,(S/N) Entrega Medicamento Ambulatorio,(S/N) Permite Osteosintesis
679,N,813007875,HOSPITALMUNICIPAL NUESTRA SEÑORADE GUADALUPE-,ESE HOSPITAL MUNICIPAL NUESTRA SEÑORA DE GUADA...,1.0,OFICINA BOGOTA,HUILA,41,GUADALUPE,31900,...,contactenos@esehosguada.gov.co,NaN,NaN,S,(8)8321052,NaN,NaN,General,S,N
1519,N,832001110,A&G SERVICIOS DE SALUD,A&G SERVICIOS DE SALUD,1.0,OFICINA BOGOTA,CUNDINAMARCA,25,ZIPAQUIRA,89900,...,gerencia@aygserviciosdesalud.com,LUN A VIE 7AM A 8PM - SAB 7AM A 1PM,CERRADO,S,(1)8521530,NaN,(1)7455901 - (1)7455902,General,N,N
1903,N,900582598,ADMINISTRADORA CLINICALA COLINA S.A.S,CLINICA LA COLINA S.A.S,3.0,OFICINA BOGOTA,BOGOTA D.C.,11,BOGOTA,100,...,julian.garcia@clinicadelcountry.com,24 HORAS,24 HORAS,S,(1)3905099,3009125057,6013904887,General,N,S
1909,N,900595618,AEPSI S.A.S,AEPSI S.A.S,1.0,OFICINA BOGOTA,BOGOTA D.C.,11,BOGOTA,100,...,info@aepsi.com.co,NaN,NaN,S,(1)2189865,3208724660,NaN,General,NaN,N
247,N,901447598,AEROMAS SAS,AEROMAS S.A.S,3.0,OFICINA BOGOTA,BOGOTA D.C.,11,BOGOTA,100,...,dircomercial@aeromas.com.co,NaN,NaN,S,NaN,3166928115,NaN,General,NaN,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42,C,37720444,VIVIANA INES CORTES BUITRAGO,VIVIANA INES CORTES BUITRAGO,2.0,OFICINA BUCARAMANGA,SANTANDER,68,BUCARAMANGA,100,...,VIVICORTES28@HOTMAIL.COM,NaN,NaN,S,(7)6989918,NaN,(7)6989918,General,NaN,N
309,N,900280774,WOUND CLINIC S.A.S.,WOUND CLINIC S.A.S,1.0,OFICINA CALI,VALLE DEL CAUCA,76,SANTIAGO DE CALI,100,...,gerencia@woundclinic.com.co,NaN,NaN,S,NaN,3504227438 casos COVID,3504227438 agudos no COVID,General,NaN,N
313,N,900280774,WOUND CLINIC S.A.S.SUCURSAL,WOUND CLINIC S.A.S,1.0,OFICINA CALI,CAUCA,19,POPAYAN,100,...,coordpopayan@woundclinic.com.co,NaN,NaN,S,NaN,3205778655,NaN,General,NaN,N
1310,N,802019855,YEPES PORTO OFTALMOLOGIA,YEPES PORTO OFTALMOLOGIA,1.0,OFICINA BARRANQUILLA,ATLANTICO,8,BARRANQUILLA,100,...,asistente@yepesporto.com,NaN,NaN,S,(5)3785945,NaN,3566557- 3562939-3563232-3563401,General,N,N


In [ ]:
df_merged.to_excel("df_merged.xlsx", index=False)
# df_unicos.to_excel('df_unicos.xlsx', index = False)